# Large neighborhood search and variants

In [ ]:
import sys, os
sys.path.append('../')

In [ ]:
from discrete_optimization.generic_tools.do_problem import get_default_objective_setup, ParamsObjectiveFunction, \
    ObjectiveHandling, ModeOptim
from discrete_optimization.generic_tools.lp_tools import ParametersMilp
from discrete_optimization.generic_tools.result_storage.result_storage import ResultStorage
from discrete_optimization.rcpsp.solver.rcpsp_lp_solver import LP_RCPSP, LP_RCPSP_Solver, LP_MRCPSP
from discrete_optimization.rcpsp.rcpsp_parser import parse_file, files_available, get_data_available
from discrete_optimization.rcpsp.rcpsp_model import RCPSPModel, UncertainRCPSPModel, \
    create_poisson_laws_duration, MethodBaseRobustification, MethodRobustification, RCPSPSolution, \
    SingleModeRCPSPModel, MultiModeRCPSPModel
from discrete_optimization.rcpsp.rcpsp_utils import plot_resource_individual_gantt, plot_ressource_view
import matplotlib.pyplot as plt
from discrete_optimization.generic_tools.lns_mip import LNS_MILP
from discrete_optimization.rcpsp.solver.rcpsp_lp_lns_solver import InitialSolutionRCPSP, \
    InitialMethodRCPSP, ConstraintHandlerFixStartTime, ConstraintHandlerStartTimeInterval, ConstraintHandlerStartTimeIntervalMRCPSP

In [ ]:
file = [f for f in files_available if 'j1201_1.sm' in f][0]
rcpsp_problem: RCPSPModel = parse_file(file)
if isinstance(rcpsp_problem, MultiModeRCPSPModel):
    rcpsp_problem.set_fixed_modes([1 for i in range(rcpsp_problem.n_jobs)])
params_objective_function = get_default_objective_setup(problem=rcpsp_problem)
params_objective_function = ParamsObjectiveFunction(objectives=["makespan"],
                                                    weights=[-1],
                                                    objective_handling=ObjectiveHandling.AGGREGATE,
                                                    sense_function=ModeOptim.MAXIMIZATION)
solver = LP_MRCPSP(rcpsp_model=rcpsp_problem,
                   lp_solver=LP_RCPSP_Solver.GRB,
                   params_objective_function=params_objective_function)
solver.init_model(greedy_start=False)
parameters_milp = ParametersMilp(time_limit=100,
                                 pool_solutions=1000,
                                 mip_gap_abs=0.001,
                                 mip_gap=0.001,
                                 retrieve_all_solution=True,
                                 n_solutions_max=100)

## Initial solution provider : 
As a first solution we need something... here we just make a proxy to dummy solution or greedy one to start with a first solution. If you can't provide a first solution, you can eventually let the frst iteration of the MIP find a feasible solution (if your problem is difficult, the time out will end the optim and you'll end up with a first suboptimal solution if the mp found one)

In [ ]:
initial_solution_provider = InitialSolutionRCPSP(problem=rcpsp_problem,
                                                 initial_method=InitialMethodRCPSP.DUMMY,
                                                 params_objective_function=params_objective_function)

## Constraint handler
The LNS needs a constraint handler object : it is an object that adds new constraint to the MILP model considering a resultStore object (typically the result found by the previous iteration). This can be any constraint you want : fixing most of variable is the most classical LNS version. For RCPSP we tested a custom constraint handler that lets a liberty on the starting time of subset of request. To make the milp easy, the liberty is limiter to [current_start_time-minus_delta, current_start_time+plus_delta]

In [ ]:
constraint_handler = ConstraintHandlerStartTimeIntervalMRCPSP(problem=rcpsp_problem,
                                                              fraction_to_fix=0.5,
                                                              minus_delta=2,
                                                              plus_delta=2)


In [ ]:
print(result_store.list_solution_fits)

## LNS : 
We're now ready to instanciate the lns mip.

In [ ]:
lns_solver = LNS_MILP(problem=rcpsp_problem,
                      milp_solver=solver,
                      initial_solution_provider=initial_solution_provider,
                      constraint_handler=constraint_handler,
                      params_objective_function=params_objective_function)
result_store = lns_solver.solve_lns(parameters_milp=parameters_milp,
                                    nb_iteration_lns=100,
                                    skip_first_iteration=False)
solution, fit = result_store.get_best_solution_fit()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot([-x[1] for x in result_store.list_solution_fits])
ax.set_title('Evolution of makespan during the LNS algorithm')